# Kyra — Target-T4 validation runner (Kaggle free T4 GPU)

Decision D-015 assigns this notebook as the project's target-T4 execution
environment: Kaggle provisions a free NVIDIA T4 GPU ("Settings > Accelerator >
GPU T4"). It runs the repository's frozen checkout and records the evidence
that the assignment's target-T4 validation requires.

**What this notebook does**

1. Identifies the environment (Python version, GPU via `nvidia-smi`).
2. Gets the repository files onto the machine.
3. Installs the frozen core environment from `requirements.lock` (CPU-only,
   deterministic core; ~1 minute).
4. Runs the full 189-test suite (dated 2026-09-07; the count is a *dated*
   snapshot — record the actual number printed, never edit the number).
5. Runs the Person 1 seven-check verifier `scripts/verify_person1.py`.
6. Audits that the frozen golden pack regenerates byte-identically
   (SHA-256 inventory match).
7. Optionally (flag below) installs the optional GPU extras from
   `requirements-gpu.lock` (torch 2.14.0 / ultralytics 8.4.138 — large
   download) and confirms `torch.cuda.is_available()` on the T4. This is a
   published-pin smoke check only, not a model-accuracy result.
8. Prints and saves the evidence report:
   `PERSON1_TEST_REPORT_T4_<date>_draft.txt` in `/kaggle/working`.

**How to use**

1. Create a Kaggle notebook (kaggle.com > Code > New Notebook) and set
   Settings > Accelerator = **GPU T4**.
2. Get the repository into the notebook: the repository is private, so the
   easiest is **Upload** the repo as a Kaggle dataset (GitHub > Code >
   Download ZIP, then New Dataset or Add-Input from your Kaggle dataset).
   The cells below use `/kaggle/input/` automatically; you can also clone
   with a fine-grained GitHub token by setting the `KYRA_GIT_URL` variable.
3. Run all cells. Copy the printed report into
   `docs/PERSON1_TEST_REPORT_T4_<date>.md` in a reviewed pull request.
4. Do **not** edit test totals or claim events that did not run. A Windows
   laptop result or a non-T4 GPU is **not** T4 evidence.

The commands mirror `docs/PERSON1_HANDOFF.md` steps 1-2.


In [ ]:
import os
import pathlib
import platform
import shutil
import subprocess
import sys
import time

WORKDIR = pathlib.Path("/kaggle/working")
print("=== Environment identity ===")
print("platform:", platform.platform())
print("python:", sys.version.splitlines()[0])
print("python ver:", sys.version_info[:3])
print("cwd:", pathlib.Path.cwd())
print("nvidia-smi:")
try:
    print(
        subprocess.run(
            ["nvidia-smi"], capture_output=True, text=True
        ).stdout.strip()
    )
except Exception as exc:
    print("nvidia-smi unavailable:", exc)

# Switch on the optional GPU smoke (torch/ultralytics install, ~2 GB download).
RUN_GPU_SMOKE = False
print("GPU smoke flag (torch/ultralytics):", RUN_GPU_SMOKE)


## 1. Get the repository files

The repository is private, so Kaggle needs the files from you. Either route
works - the cell below tries both:

- **Route A (recommended):** Create a dataset from the repo ZIP
  (kaggle.com > Datasets > New Dataset > Upload; the ZIP is the GitHub
  "Download ZIP" of `DontHash/kyra-shelf-guard`). In the notebook, open the
  Input panel > **Add Input > Your Datasets** and select it, then re-run this
  cell. Kaggle auto-extracts ZIP datasets, so the notebook searches the
  extracted tree for `pyproject.toml`.
- **Route B (secret):** Notebook Settings > **Secrets > Add New Secret**:
  key `KYRA_GIT_URL`, value
  `https://<fine-grained-token>@github.com/DontHash/kyra-shelf-guard.git`
  where the token has `Contents: Read` on the repo. The cell clones it.

The cell also prints what it finds under `/kaggle/input` so a miss is
diagnosable instead of mysterious.


In [ ]:
import glob
import pathlib
import subprocess
import zipfile

WORKDIR = pathlib.Path("/kaggle/working")
REPO = WORKDIR / "kyra-repo"
if REPO.exists():
    shutil.rmtree(REPO)
found = None


def _is_repo_root(root: pathlib.Path) -> bool:
    return (root / "pyproject.toml").exists() and (
        root / "scripts" / "verify_person1.py"
    ).exists()


git_url = os.environ.get("KYRA_GIT_URL", "").strip()
if git_url:
    print("cloning from KYRA_GIT_URL...")
    subprocess.run(
        ["git", "clone", "--depth", "1", git_url, str(REPO)],
        check=True,
    )
    found = REPO if _is_repo_root(REPO) else None

if found is None and pathlib.Path("/kaggle/input").exists():
    print("listing /kaggle/input:")
    try:
        for entry in sorted(os.listdir("/kaggle/input")):
            print("  -", entry)
    except OSError as exc:
        print("  list failed:", exc)
    for cand in pathlib.Path("/kaggle/input").rglob("pyproject.toml"):
        root = cand.parent
        if _is_repo_root(root):
            found = root
            break

if found is None:
    zips = sorted(glob.glob("/kaggle/input/**/*.zip", recursive=True))
    print("zips found under /kaggle/input:", zips)
    if zips:
        dest = WORKDIR / "src-zip"
        dest.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(zips[0]) as zf:
            zf.extractall(dest)
        for cand in dest.rglob("pyproject.toml"):
            root = cand.parent
            if _is_repo_root(root):
                found = root
                break

if found is None:
    raise SystemExit(
        "No repository found under /kaggle/input and no KYRA_GIT_URL set.\n"
        "Fix (either):\n"
        "  1) Upload the repo ZIP as a Kaggle dataset, then in the notebook "
        "Input panel: Add Input > Your Datasets > select it, and re-run.\n"
        "  2) Settings > Secrets: add KYRA_GIT_URL = "
        "https://<token>@github.com/DontHash/kyra-shelf-guard.git "
        "(token needs Contents: Read on the repo), and re-run."
    )

if found != REPO:
    if REPO.exists():
        shutil.rmtree(REPO)
    shutil.move(str(found), str(REPO))

print("repo root:", REPO)
print("has pyproject.toml:", (REPO / "pyproject.toml").exists())
print(
    "has scenario-fixtures/golden:",
    (REPO / "scenario-fixtures" / "golden").exists(),
)


## 2. Install the frozen core environment

This matches `docs/PERSON1_HANDOFF.md` step 1 (requirements.lock install and
editable package install) and exercises the real locked pins.


In [ ]:
import subprocess
import sys

print("install core lock...")
r1 = subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "-r",
        str(REPO / "requirements.lock"),
    ],
    capture_output=True,
    text=True,
)
print("pip install -r requirements.lock exit:", r1.returncode)
r2 = subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-e", str(REPO)],
    capture_output=True,
    text=True,
)
print("pip install -e . exit:", r2.returncode)
if r1.returncode or r2.returncode:
    print(r1.stderr[-2000:])
    print(r2.stderr[-2000:])
    raise SystemExit("Core environment install failed.")


In [ ]:
import cv2
import numpy as np
import pydantic

print("numpy", np.__version__, "| opencv", cv2.__version__, "| pydantic", pydantic.VERSION)


## 3. Full test suite (dated total)

Run the complete suite from the repository root. The expected dated total on
2026-09-07 is 189; record the **actual** number printed in this run, and do
not restate the number in any summary as anything other than a dated total.


In [ ]:
t0 = time.time()
r = subprocess.run([sys.executable, "-m", "pytest", "-q"], cwd=REPO, capture_output=True, text=True)
elapsed = time.time() - t0
out = (r.stdout + r.stderr).strip().splitlines()
print("pytest exit:", r.returncode, "| elapsed:", round(elapsed, 2), "s")
print("\n".join(out[-6:]))
PYTEST_PASSED = r.returncode == 0
PYTEST_TAIL = out[-1] if out else "(no output)"
if not PYTEST_PASSED:
    raise SystemExit("Full suite failed on the target environment.")


## 4. Person 1 seven-check verifier

`scripts/verify_person1.py` runs the schema, golden-checksum, reader,
baseline, MiDaS-policy, and provenance checks. It must exit 0.


In [ ]:
r = subprocess.run([sys.executable, str(REPO / "scripts" / "verify_person1.py")],
                    cwd=REPO, capture_output=True, text=True)
print("verify_person1 exit:", r.returncode)
print((r.stdout + r.stderr).strip()[-3000:])
VERIFY_PASSED = r.returncode == 0
if not VERIFY_PASSED:
    raise SystemExit("Seven-check verifier failed on the target environment.")


## 5. Golden pack byte-identity audit

Regenerate the golden pack into a scratch directory and compare the SHA-256
inventory against the committed `scenario-fixtures/golden/checksums.sha256`.
Byte-identical regeneration is the reproducibility claim; any mismatch fails
the audit.


In [ ]:
import shutil

audit = WORKDIR / "golden-audit"
if audit.exists():
    shutil.rmtree(audit)
audit.mkdir()
r = subprocess.run(
    [
        sys.executable,
        str(REPO / "scripts" / "generate_golden_pack.py"),
        str(audit),
    ],
    cwd=REPO,
    capture_output=True,
    text=True,
)
print("generate_golden_pack exit:", r.returncode)
committed = sorted(
    (REPO / "scenario-fixtures" / "golden" / "checksums.sha256")
    .read_text()
    .splitlines()
)
regenerated = sorted((audit / "checksums.sha256").read_text().splitlines())
GOLDEN_IDENTICAL = committed == regenerated
print("committed entries:", len(committed), "| regenerated entries:", len(regenerated))
print("golden pack byte-identical:", GOLDEN_IDENTICAL)
if not (GOLDEN_IDENTICAL and r.returncode == 0):
    raise SystemExit("Golden pack regeneration audit failed.")


## 6. Optional GPU smoke (publishes torch/ultralytics pins only)

If `RUN_GPU_SMOKE = True` in the first cell, this installs the optional GPU
extras (`requirements-gpu.lock`) and checks CUDA availability and the GPU
model. This is a published-pin smoke check for Gate 7 planning, not a model
accuracy or end-to-end result.


In [ ]:
if RUN_GPU_SMOKE:
    r = subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "-r",
            str(REPO / "requirements-gpu.lock"),
        ],
        capture_output=True,
        text=True,
    )
    print("pip install -r requirements-gpu.lock exit:", r.returncode)
    if r.returncode:
        print(r.stderr[-2000:])
        raise SystemExit("GPU extras install failed.")
    probe_lines = [
        "import torch",
        "print('torch', torch.__version__)",
        "print('cuda', torch.cuda.is_available())",
        "name = 'none'",
        "if torch.cuda.is_available():",
        "    name = torch.cuda.get_device_name(0)",
        "print('device', name)",
    ]
    probe_code = "\n".join(probe_lines)
    r = subprocess.run(
        [sys.executable, "-c", probe_code],
        cwd=REPO,
        capture_output=True,
        text=True,
    )
    print("torch probe exit:", r.returncode)
    print(r.stdout.strip(), r.stderr.strip()[-500:])
    GPU_SMOKE_OK = r.returncode == 0
else:
    print("GPU smoke skipped (RUN_GPU_SMOKE = False).")
    GPU_SMOKE_OK = None


## 7. Evidence report

Assembles the dated report and saves it to `/kaggle/working`. Copy the text
into `docs/PERSON1_TEST_REPORT_T4_<date>.md` in a reviewed pull request.
Date totals and exits printed here are the evidence; do not alter them.


In [ ]:
from datetime import date

report = []
report.append("# Person 1 Target-T4 Test Report (Kaggle free T4 GPU)")
report.append("")
report.append(
    f"- **Date:** {date.today().isoformat()} "
    "(dated evidence; totals are a snapshot of this run)"
)
report.append(
    f"- **Environment:** Kaggle notebook; Python "
    f"{sys.version_info.major}.{sys.version_info.minor}.{sys.version_info.micro}; "
    f"platform {platform.platform()}"
)
report.append(
    "- **Repository commit:** see the checkout used "
    "(record the SHA of the frozen main here at run time)"
)
report.append(
    "- **GPU:** see nvidia-smi output above "
    "(must be NVIDIA T4 for target-T4 evidence)"
)
report.append("")
report.append("## Commands and results")
report.append("")
pytest_exit = 0 if PYTEST_PASSED else 1
verify_exit = 0 if VERIFY_PASSED else 1
report.append("1. `pip install -r requirements.lock` — exit 0")
report.append("2. `pip install -e .` — exit 0")
report.append(f"3. `python -m pytest -q` — exit {pytest_exit}; {PYTEST_TAIL}")
report.append(f"4. `python scripts/verify_person1.py` — exit {verify_exit}")
report.append(
    "5. golden-pack regeneration audit — "
    f"byte-identical: {GOLDEN_IDENTICAL}"
)
report.append(
    "6. GPU smoke (optional) — "
    f"run: {RUN_GPU_SMOKE}; ok: {GPU_SMOKE_OK}"
)
report.append("")
report.append("## Honest limits")
report.append("")
report.append(
    "- This evidence is a labelled target-environment run of the committed "
    "suite only; it does not prove real end-to-end detection."
)
report.append(
    "- The optional GPU pins are published-pin smoke evidence, "
    "not model accuracy."
)
report.append("")
text = "\n".join(report)
report_name = "PERSON1_TEST_REPORT_T4_" f"{date.today().isoformat()}_draft.txt"
report_path = WORKDIR / report_name
report_path.write_text(text)
print(text)
print("")
print("saved:", report_path)


## After running

1. Open a pull request adding `docs/PERSON1_TEST_REPORT_T4_<date>.md` with the
   report text plus the environment identity and the checkout SHA.
2. Reference this notebook and decisions D-014/D-015.
3. Do not restate dated totals as current, and do not claim events that did
   not actually run.
